# Pipeline Test Notebook
Test the `utils.py` S3 helpers and `pipeline.py` parse + upsert functions end-to-end:
1. List available dumps in S3
2. Download one dump locally
3. Parse it with `parse_dump()`
4. Inspect the aggregates
5. Clean up the temp file
6. **Upsert first 3 dumps into Postgres** via `upsert_aggregates()`
7. Compute and verify `artist_daily_stats` / `track_daily_stats` via `compute_daily_stats()`

In [1]:
import importlib
import sys
from pathlib import Path

# Add parent dir to path so we can import local modules
sys.path.insert(0, str(Path.cwd().parent))

# Force-reload to pick up any changes since the kernel started
import pipeline, utils
importlib.reload(utils)
importlib.reload(pipeline)

from utils import get_s3_client, list_s3_artifacts, download_s3_dump, connect_postgres, load_db_credentials
from pipeline import parse_dump, upsert_aggregates, compute_daily_stats

print("Imports OK")

Imports OK


In [2]:
import os
from dotenv import load_dotenv

# Load AWS creds from the project .env
env_path = Path.cwd().parent.parent / ".env"
print(f"Loading env from: {env_path}  (exists={env_path.exists()})")
load_dotenv(dotenv_path=env_path, override=True)

# Verify AWS creds are set
for k in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN", "AWS_DEFAULT_REGION"]:
    v = os.getenv(k)
    print(f"  {k} = {'set (' + v[:8] + '...)' if v else 'NOT SET'}")

Loading env from: /Users/didiermunezero/Documents/NU/Junior/DE 300/de300-2026wi-stall/project/.env  (exists=True)
  AWS_ACCESS_KEY_ID = set (ASIAYAAO...)
  AWS_SECRET_ACCESS_KEY = set (Hv350r1D...)
  AWS_SESSION_TOKEN = set (IQoJb3Jp...)
  AWS_DEFAULT_REGION = set (us-east-...)


In [3]:
# Force a fresh boto3 session that re-reads env vars
import boto3
boto3.DEFAULT_SESSION = None
session = boto3.Session()

sts = session.client("sts")
identity = sts.get_caller_identity()
print(f"Authenticated as: {identity['Arn']}")
print(f"Account: {identity['Account']}")

ClientError: An error occurred (ExpiredToken) when calling the GetCallerIdentity operation: The security token included in the request is expired

In [ ]:
BUCKET = "stall-munezero-final-project"

# Use a fresh client from the reset session
s3 = session.client("s3")

# Test list_s3_artifacts (pass explicit client to bypass get_s3_client's cached session)
keys = list_s3_artifacts(BUCKET, "listenbrainz/incremental/", s3_client=s3)
print(f"✓ list_s3_artifacts found {len(keys)} keys:")
for k in keys:
    print(f"  {k}")

✓ list_s3_artifacts found 41 keys:
  listenbrainz/incremental/listenbrainz-listens-dump-2400-20260118-000003-incremental.tar.zst
  listenbrainz/incremental/listenbrainz-listens-dump-2401-20260119-000003-incremental.tar.zst
  listenbrainz/incremental/listenbrainz-listens-dump-2402-20260120-000003-incremental.tar.zst
  listenbrainz/incremental/listenbrainz-listens-dump-2403-20260120-200718-incremental.tar.zst
  listenbrainz/incremental/listenbrainz-listens-dump-2404-20260121-000003-incremental.tar.zst
  listenbrainz/incremental/listenbrainz-listens-dump-2405-20260122-000003-incremental.tar.zst
  listenbrainz/incremental/listenbrainz-listens-dump-2406-20260123-000003-incremental.tar.zst
  listenbrainz/incremental/listenbrainz-listens-dump-2407-20260124-000003-incremental.tar.zst
  listenbrainz/incremental/listenbrainz-listens-dump-2408-20260125-000003-incremental.tar.zst
  listenbrainz/incremental/listenbrainz-listens-dump-2409-20260126-000003-incremental.tar.zst
  listenbrainz/incrementa

In [ ]:
# Download a dump from S3 and parse it
# Pick the first key from the list
dump_key = keys[0]
print(f"Downloading: {dump_key}")

local_path = download_s3_dump(BUCKET, dump_key, local_dir=Path("./tmp_test"), s3_client=s3)
print(f"Local path: {local_path}")
print(f"Size: {local_path.stat().st_size / 1e6:.1f} MB")

Downloading: listenbrainz/incremental/listenbrainz-listens-dump-2400-20260118-000003-incremental.tar.zst
Done (184.4 MB)
Local path: tmp_test/listenbrainz-listens-dump-2400-20260118-000003-incremental.tar.zst
Size: 184.4 MB


In [ ]:
# Parse the downloaded S3 dump (cap at 50k lines for quick test)
track_daily, artist_daily, track_info, artist_info, summary = parse_dump(local_path, max_lines=50_000)

print("Parse summary:")
for k, v in summary.items():
    print(f"  {k}: {v:,}")

Parsing listens: 50000it [00:00, 173242.46it/s]


Parse summary:
  lines_parsed: 50,001
  bad_json: 0
  missing_timestamp: 0
  missing_artist_mbid: 0
  unique_track_day_keys: 43,843
  unique_artist_day_keys: 1,265
  unique_tracks: 28,150
  unique_artists: 1,160


In [ ]:
# Cleanup temp file
local_path.unlink()
print(f"Deleted: {local_path}")

# Remove temp dir if empty
import shutil
tmp_dir = Path("./tmp_test")
if tmp_dir.exists() and not any(tmp_dir.iterdir()):
    tmp_dir.rmdir()
    print(f"Removed empty dir: {tmp_dir}")

Deleted: tmp_test/listenbrainz-listens-dump-2400-20260118-000003-incremental.tar.zst
Removed empty dir: tmp_test


## 6) Upsert first 3 dumps into Postgres
Downloads each dump from S3, parses it fully, upserts aggregates, then cleans up the temp file before moving to the next.

In [ ]:
# Pick the first 3-10 dumps from S3
dumps_to_ingest = keys[5:7]
print(f"Will ingest {len(dumps_to_ingest)} dumps:")
for k in dumps_to_ingest:
    print(f"  {k}")

Will ingest 2 dumps:
  listenbrainz/incremental/listenbrainz-listens-dump-2405-20260122-000003-incremental.tar.zst
  listenbrainz/incremental/listenbrainz-listens-dump-2406-20260123-000003-incremental.tar.zst


In [ ]:
# # One-time schema migration: keep existing aggregates as legacy rows,
# # then switch daily tables to per-dump keys.
# migration_sql = '''
# ALTER TABLE artist_daily_listens ADD COLUMN IF NOT EXISTS dump_id TEXT;
# ALTER TABLE track_daily_listens ADD COLUMN IF NOT EXISTS dump_id TEXT;

# UPDATE artist_daily_listens SET dump_id = 'legacy' WHERE dump_id IS NULL;
# UPDATE track_daily_listens SET dump_id = 'legacy' WHERE dump_id IS NULL;

# ALTER TABLE artist_daily_listens ALTER COLUMN dump_id SET NOT NULL;
# ALTER TABLE track_daily_listens ALTER COLUMN dump_id SET NOT NULL;

# ALTER TABLE artist_daily_listens DROP CONSTRAINT IF EXISTS artist_daily_listens_pkey;
# ALTER TABLE track_daily_listens DROP CONSTRAINT IF EXISTS track_daily_listens_pkey;

# ALTER TABLE artist_daily_listens
#     ADD CONSTRAINT artist_daily_listens_pkey PRIMARY KEY (day, artist_mbid, dump_id);
# ALTER TABLE track_daily_listens
#     ADD CONSTRAINT track_daily_listens_pkey PRIMARY KEY (day, recording_id, dump_id);

# CREATE INDEX IF NOT EXISTS idx_artist_daily_dump ON artist_daily_listens(dump_id);
# CREATE INDEX IF NOT EXISTS idx_track_daily_dump ON track_daily_listens(dump_id);
# '''

# conn = connect_postgres()
# conn.autocommit = True
# with conn.cursor() as cur:
#     cur.execute(migration_sql)
# conn.close()

# print('Schema migration complete: daily tables now keyed by (day, entity, dump_id).')

In [ ]:
import time

tmp_dir = Path("./tmp_test")

for i, dump_key in enumerate(dumps_to_ingest, 1):
    print(f"\n{'='*60}")
    print(f"[{i}/{len(dumps_to_ingest)}] {dump_key}")
    print(f"{'='*60}")

    # Download
    t0 = time.perf_counter()
    local_path = download_s3_dump(BUCKET, dump_key, local_dir=tmp_dir, s3_client=s3)

    # Parse (full — no line limit)
    track_daily, artist_daily, track_info, artist_info, summary = parse_dump(local_path)
    print("Parse summary:")
    for k, v in summary.items():
        print(f"  {k}: {v:,}")

    # Upsert
    conn = connect_postgres()
    upsert_aggregates(conn, track_daily, artist_daily, track_info, artist_info, dump_path=dump_key)
    conn.close()

    # Cleanup
    local_path.unlink()
    elapsed = time.perf_counter() - t0
    print(f"Done in {elapsed:.1f}s — temp file deleted.")

# Remove temp dir if empty
if tmp_dir.exists() and not any(tmp_dir.iterdir()):
    tmp_dir.rmdir()
    print(f"\nRemoved empty dir: {tmp_dir}")

print("\nAll 3 dumps ingested.")

In [ ]:
# Verify: check row counts, dump-level lineage, and ingestion_state
conn = connect_postgres()
with conn.cursor() as cur:
    for table in ["artist_info", "track_info", "artist_daily_listens", "track_daily_listens"]:
        cur.execute(f"SELECT COUNT(*) FROM {table};")  # noqa: S608 — table names are hardcoded
        print(f"  {table}: {cur.fetchone()[0]:,} rows")

    print("\nartist_daily_listens rows by dump_id:")
    cur.execute(
        """
        SELECT dump_id, COUNT(*) AS rows, SUM(listen_count) AS listens
        FROM artist_daily_listens
        GROUP BY dump_id
        ORDER BY dump_id;
        """
    )
    for dump_id, rows, listens in cur.fetchall():
        print(f"  {dump_id}: rows={rows:,}, listens={int(listens):,}")

    print("\ntrack_daily_listens rows by dump_id:")
    cur.execute(
        """
        SELECT dump_id, COUNT(*) AS rows, SUM(listen_count) AS listens
        FROM track_daily_listens
        GROUP BY dump_id
        ORDER BY dump_id;
        """
    )
    for dump_id, rows, listens in cur.fetchall():
        print(f"  {dump_id}: rows={rows:,}, listens={int(listens):,}")

    cur.execute("SELECT last_dump_id, last_dump_path, loaded_at FROM ingestion_state WHERE id = 1;")
    row = cur.fetchone()
    print(f"\ningestion_state:")
    print(f"  last_dump_id:   {row[0]}")
    print(f"  last_dump_path: {row[1]}")
    print(f"  loaded_at:      {row[2]}")
conn.close()

  artist_info: 54,763 rows
  track_info: 2,363,162 rows


In [ ]:
import time

In [ ]:
def ingest_next_n_dumps(n_dumps: int, bucket: str = "stall-munezero-final-project", 
                        prefix: str = "listenbrainz/incremental/", s3_client=None) -> dict:
    """
    Ingest the next N dumps from S3 after the last dump recorded in ingestion_state.
    
    Parameters
    ----------
    n_dumps : int
        Number of dumps to ingest (after last_dump_id).
    bucket : str
        S3 bucket name.
    prefix : str
        S3 prefix for dump path.
    s3_client : boto3 S3 client, optional
        If None, uses the session client from the notebook.
    
    Returns
    -------
    dict
        Summary with keys: last_dump_id, n_ingested, dumps_ingested, total_rows_parsed.
    """
    import re
    from pipeline import _load_alias_map_from_db
    
    if s3_client is None:
        s3_client = session.client("s3")
    
    # Get last_dump_id from database
    conn = connect_postgres()
    with conn.cursor() as cur:
        cur.execute("SELECT last_dump_id FROM ingestion_state WHERE id = 1;")
        row = cur.fetchone()
        last_dump_id = int(row[0]) if row and row[0] else 0
    conn.close()
    print(f"Last ingested dump_id: {last_dump_id}")
    
    # List all dumps from S3
    all_keys = list_s3_artifacts(bucket, prefix, s3_client=s3_client)
    print(f"Found {len(all_keys)} total dumps in S3")
    
    # Filter to dumps with ID > last_dump_id
    def extract_dump_id(key: str) -> int:
        m = re.search(r"dump-(\d+)-", key)
        return int(m.group(1)) if m else 0
    
    candidate_keys = [k for k in all_keys if extract_dump_id(k) > last_dump_id]
    candidate_keys.sort(key=extract_dump_id)
    
    if not candidate_keys:
        print("No new dumps to ingest.")
        return {"last_dump_id": last_dump_id, "n_ingested": 0, "dumps_ingested": [], "total_rows_parsed": 0}
    
    dumps_to_ingest = candidate_keys[:n_dumps]
    print(f"Will ingest {len(dumps_to_ingest)} dumps (next after {last_dump_id}):")
    for k in dumps_to_ingest:
        print(f"  {k}")
    
    # Ingest each dump
    tmp_dir = Path("./tmp_batch_ingest")
    total_rows_parsed = 0
    dumps_ingested = []
    alias_conn = connect_postgres()
    
    try:
        for i, dump_key in enumerate(dumps_to_ingest, 1):
            dump_id = extract_dump_id(dump_key)
            print(f"\n{'='*60}")
            print(f"[{i}/{len(dumps_to_ingest)}] dump_id={dump_id}: {dump_key}")
            print(f"{'='*60}")
            
            t0 = time.perf_counter()
            
            # Refresh alias map before each dump so canonical IDs from prior
            # dumps in this batch are reused in later dumps.
            alias_to_mbid = _load_alias_map_from_db(alias_conn)

            # Download
            local_path = download_s3_dump(bucket, dump_key, local_dir=tmp_dir, s3_client=s3_client)
            
            # Parse
            track_daily, artist_daily, track_info, artist_info, summary = parse_dump(
                local_path,
                alias_to_mbid=alias_to_mbid,
            )
            total_rows_parsed += summary["lines_parsed"]
            print("Parse summary:")
            for k, v in summary.items():
                print(f"  {k}: {v:,}")
            
            # Upsert
            conn = connect_postgres()
            upsert_aggregates(conn, track_daily, artist_daily, track_info, artist_info, dump_path=dump_key)
            conn.close()
            
            # Cleanup
            local_path.unlink()
            elapsed = time.perf_counter() - t0
            print(f"Done in {elapsed:.1f}s — temp file deleted.")
            dumps_ingested.append(dump_id)
    
    finally:
        alias_conn.close()
        # Clean up temp dir if empty
        if tmp_dir.exists() and not any(tmp_dir.iterdir()):
            tmp_dir.rmdir()
            print(f"\nRemoved empty dir: {tmp_dir}")
    
    print(f"\n{'='*60}")
    print(f"Batch ingest complete: {len(dumps_ingested)} dumps ({total_rows_parsed:,} rows total)")
    print(f"{'='*60}")
    
    return {
        "last_dump_id": last_dump_id,
        "n_ingested": len(dumps_ingested),
        "dumps_ingested": dumps_ingested,
        "total_rows_parsed": total_rows_parsed,
    }

# Example usage (uncomment to use):
# result = ingest_next_n_dumps(n_dumps=3)
# print(f"Ingested: {result}")
print("Function 'ingest_next_n_dumps' defined. Call with ingest_next_n_dumps(n_dumps=N) to ingest next N dumps.")

Function 'ingest_next_n_dumps' defined. Call with ingest_next_n_dumps(n_dumps=N) to ingest next N dumps.


In [ ]:
ingest_next_n_dumps(n_dumps=1)

Last ingested dump_id: 2404
Found 41 total dumps in S3
Will ingest 1 dumps (next after 2404):
  listenbrainz/incremental/listenbrainz-listens-dump-2405-20260122-000003-incremental.tar.zst

[1/1] dump_id=2405: listenbrainz/incremental/listenbrainz-listens-dump-2405-20260122-000003-incremental.tar.zst
Done (265.0 MB)


Parsing listens: 5685007it [00:46, 123024.25it/s]


Parse summary:
  lines_parsed: 5,685,007
  bad_json: 0
  missing_timestamp: 0
  missing_artist_mbid: 0
  unique_track_day_keys: 4,276,253
  unique_artist_day_keys: 16,906
  unique_tracks: 793,063
  unique_artists: 16,400
  Upserting artist_info (16,400 rows)... done (5.4s)
  Upserting track_info (793,063 rows)... done (294.6s)
  Upserting track_daily_listens (4,276,253 rows)... done (1999.5s)
  Upserting artist_daily_listens (16,906 rows)... done (7.7s)
Upserts complete.
Done in 2390.2s — temp file deleted.

Removed empty dir: tmp_batch_ingest

Batch ingest complete: 1 dumps (5,685,007 rows total)


{'last_dump_id': 2404,
 'n_ingested': 1,
 'dumps_ingested': [2405],
 'total_rows_parsed': 5685007}

In [ ]:
# Build daily stats tables from the ingested listens tables
import importlib
import os
from pathlib import Path
import pipeline

# Ensure Windows Spark can start (HADOOP_HOME/hadoop.home.dir must exist before JVM boot).
if os.name == "nt":
    hadoop_home = Path.home() / ".hadoop"
    (hadoop_home / "bin").mkdir(parents=True, exist_ok=True)
    os.environ["HADOOP_HOME"] = str(hadoop_home)
    os.environ["hadoop.home.dir"] = str(hadoop_home)
    print(f"HADOOP_HOME set to: {hadoop_home}")

# Reload pipeline so notebook picks up latest compute_daily_stats changes.
importlib.reload(pipeline)

creds = load_db_credentials()
print("Computing daily stats (artist + track)...")
pipeline.compute_daily_stats(creds)

# Verify stats tables now contain rows
conn = connect_postgres()
with conn.cursor() as cur:
    for table in ["artist_daily_stats", "track_daily_stats"]:
        cur.execute(f"SELECT COUNT(*) FROM {table};")  # noqa: S608 — table names are hardcoded
        print(f"  {table}: {cur.fetchone()[0]:,} rows")
conn.close()

HADOOP_HOME set to: C:\Users\Natha\.hadoop
Computing daily stats (artist + track)...
Loaded 45,261 rows from artist_daily_listens
Wrote 45,261 rows to artist_daily_stats
Loaded 6,815,895 rows from track_daily_listens
Wrote 6,815,895 rows to track_daily_stats
Spark session stopped.
  artist_daily_stats: 45,261 rows
  track_daily_stats: 6,815,895 rows
